<a href="https://colab.research.google.com/github/sashaprovorova/ChemAI-Predict-the-Cure/blob/main/ChemAI_PredicttheCure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ChemAI: Predict the Cure

ДПО Команда 3:
- Дарья Жукова
- Андрей Андреенко
- Александра Проворова

## Цель проекта

Химики предоставили данные о свойствах молекул и их биологической активности против вируса гриппа. Наша задача — построить модели, способные предсказывать эффективность новых соединений.

Для каждого из которых нужно предсказать три показателя:

- `IC50` — — концентрация, при которой вещество подавляет 50% активности вируса

- `CC50` — концентрация, при которой вещество токсично для 50% клеток

- `SI` — индекс селективности, равный отношению `CC50 / IC50`

Если `SI > 8`, соединение потенциально может быть перспективным кандидатом для лекарства

## Общий pipeline решения

### Шаг 1. Загрузка данных

Загружаем библиотеки

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

Загружаем train/test данные

In [ ]:
train_url = 'https://drive.google.com/uc?export=download&id=159PZX3X5rpUO-WbzWyC9whnc8B4mNqJl'
test_url = 'https://drive.google.com/uc?export=download&id=1Ui2t87X3in-Wu-pnjkDXa_VtPsVafi0l'
submission_url = 'https://drive.google.com/uc?export=download&id=1LL6moSzpUVxJUTMeXihWvUxBJNjvj6EH'

In [ ]:
train_df = pd.read_csv(train_url)
test_df = pd.read_csv(test_url)
sample_submission = pd.read_csv(submission_url)

Проверяем размер таблиц и базовую информацию о данных в них

In [ ]:
train_df.shape

(751, 214)

In [ ]:
test_df.shape

(250, 211)

In [ ]:
train_df.head(2)

,index,"IC50, mM","CC50, mM",SI,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,0,102.414420,95.757483,0.935,5.466584,5.466584,0.719259,0.719259,0.681165,18.307692,...,1,0,0,0,0,0,0,0,0,0
1,1,0.044333,8.401080,189.500,11.492712,11.492712,0.012350,-3.798024,0.769122,27.652174,...,0,1,0,0,0,0,0,0,0,0


In [ ]:
test_df.head(2)

,index,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,0,13.761882,13.761882,0.121946,-0.962625,0.770057,30.580645,450.541,432.397,450.070799,...,1,0,0,0,0,0,0,1,0,0
1,1,13.224489,13.224489,0.066132,-1.801871,0.278628,25.687500,448.380,428.220,448.100561,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 751 entries, 0 to 750
Columns: 214 entries, index to fr_urea
dtypes: float64(107), int64(107)
memory usage: 1.2 MB


In [ ]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Columns: 211 entries, index to fr_urea
dtypes: float64(104), int64(107)
memory usage: 412.2 KB


In [ ]:
train_df.describe()

,index,"IC50, mM","CC50, mM",SI,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
count,751.000000,751.000000,751.000000,751.000000,751.000000,751.000000,751.000000,751.000000,751.000000,751.000000,...,751.000000,751.000000,751.000000,751.000000,751.000000,751.000000,751.0,751.000000,751.000000,751.000000
mean,375.000000,204.544021,577.426098,89.153313,10.860070,10.860070,0.180064,-0.971890,0.577938,29.588010,...,0.055925,0.013316,0.010652,0.001332,0.001332,0.054594,0.0,0.069241,0.182423,0.006658
std,216.939316,370.367937,641.515163,788.882198,3.347314,3.347314,0.169163,1.594491,0.214190,12.713195,...,0.272400,0.114699,0.102728,0.036491,0.036491,0.227337,0.0,0.254033,1.227468,0.081377
min,0.000000,0.003517,0.700808,0.011489,2.321942,2.321942,0.000039,-6.992796,0.059567,9.545455,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
25%,187.500000,13.222351,99.998894,1.500000,8.921032,8.921032,0.048473,-1.333831,0.442842,18.306020,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
50%,375.000000,44.069306,376.580899,4.000000,12.197500,12.197500,0.121372,-0.419485,0.636477,29.281250,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
75%,562.500000,206.787402,877.508784,17.372463,13.214245,13.214245,0.290990,0.072488,0.742483,38.875000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
max,750.000000,4095.188563,4538.976189,15620.600000,15.933463,15.933463,1.374614,1.374614,0.947265,60.272727,...,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,1.000000,20.000000,1.000000


In [ ]:
test_df.describe()

,index,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
count,250.000000,250.000000,250.000000,250.000000,250.000000,250.000000,250.000000,250.000000,250.000000,250.000000,...,250.000000,250.000000,250.000000,250.0,250.0,250.000000,250.0,250.000000,250.00000,250.000000
mean,124.500000,10.746226,10.746226,0.182532,-0.953260,0.587845,29.187523,335.810320,312.690832,335.501830,...,0.048000,0.008000,0.004000,0.0,0.0,0.044000,0.0,0.072000,0.27200,0.008000
std,72.312977,3.189955,3.189955,0.167665,1.571574,0.206474,12.852043,119.724941,113.678440,119.612128,...,0.214195,0.089263,0.063246,0.0,0.0,0.205507,0.0,0.259006,1.17795,0.089263
min,0.000000,2.599352,2.599352,0.001173,-6.992796,0.079916,9.416667,151.253000,134.117000,151.136100,...,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.00000,0.000000
25%,62.250000,9.887594,9.887594,0.053571,-1.333827,0.472607,18.930233,251.667000,239.177000,251.479268,...,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.00000,0.000000
50%,124.500000,11.882133,11.882133,0.136327,-0.419785,0.632086,29.817547,305.378500,283.689500,305.163497,...,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.00000,0.000000
75%,186.750000,13.022136,13.022136,0.291678,0.052681,0.737375,38.125000,399.250750,366.158250,398.968178,...,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.00000,0.000000
max,249.000000,15.933463,15.933463,0.908843,0.908843,0.905063,60.272727,758.635000,720.331000,758.190558,...,1.000000,1.000000,1.000000,0.0,0.0,1.000000,0.0,1.000000,9.00000,1.000000



Проверяем названия колонок

In [ ]:
# train_df.columns.tolist()
# test_df.columns.tolist()

In [ ]:
train_columns = set(train_df.columns)
test_columns = set(test_df.columns)

In [ ]:
print("Колонки, которые есть только в train:", train_columns - test_columns)
print("\nКолонки, которые есть только в test:", test_columns - train_columns)

Колонки, которые есть только в train: {'SI', 'CC50, mM', 'IC50, mM'}

Колонки, которые есть только в test: set()


Отделяем признаки от целевых переменных: `IC50`, `CC50`, `SI`

In [ ]:
TARGET_COLUMNS = ['IC50, mM', 'CC50, mM', 'SI']

# признаки (features)
X = train_df.drop(columns=TARGET_COLUMNS)

# целевые переменные (targets)
y = train_df[TARGET_COLUMNS]

# test данные без target-переменных
X_test = test_df.copy()

In [ ]:
print("Размер X:", X.shape)
print("Размер y:", y.shape)
print("Размер X_test:", X_test.shape)

Размер X: (751, 211)
Размер y: (751, 3)
Размер X_test: (250, 211)


### Шаг 2. Первичный анализ данных

Проверяем:

- типы данных

- пропуски

- дубликаты

- выбросы

- распределения целевых переменных

- корреляции между признаками и таргетами

### Шаг 3. Очистка данных

Возможные действия:

- удалить дубликаты

- обработать пропуски

- привести типы данных

- убрать явно неинформативные признаки

- проверить экстремальные значения

### Шаг 4. EDA

Нужно понять:

- как распределены `IC50`, `CC50`, `SI`

- есть ли сильные выбросы

- какие признаки сильнее связаны с таргетами

- есть ли признаки с почти нулевой дисперсией

- есть ли сильно коррелирующие признаки

### Шаг 5. Baseline model

Сначала строим простую модель, чтобы получить первую точку сравнения.

Возможные baseline-модели:

- Linear Regression

- Random Forest

- CatBoost / LightGBM / XGBoost

- MultiOutputRegressor, если предсказываем сразу 3 таргета


### Шаг 6. Feature engineering

Возможные улучшения:

- масштабирование числовых признаков

- удаление слишком коррелирующих признаков

- логарифмирование таргетов, если распределения сильно скошены

- создание `SI = CC50 / IC50`, если решим предсказывать только `IC50` и `CC50`

### Шаг 7. Обучение моделей

Пробуем несколько подходов:

- отдельная модель для каждого таргета

- одна multi-output модель

- сравнение разных алгоритмов

- подбор гиперпараметров

### Шаг 8. Валидация

Используем:

- train/validation split

- cross-validation

- RMSE по каждому таргету

- средний RMSE


### Шаг 9. Предсказание для Kaggle

- обучить лучшую модель на всех train-данных

- сделать предсказания для test

- сформировать файл submission в нужном формате

- загрузить на Kaggle


### Шаг 10. Итоговый анализ

В конце фиксируем:

- какая модель дала лучший результат

- какие признаки/подходы помогли

- что не улучшило качество

- какие ограничения есть у решения

- рекоменлации того, что можно улучшить дальше